In [0]:
branches_table_path = dbutils.widgets.get("branches_table_path")
branches_grp_branches_path = dbutils.widgets.get("branches_grp_branches_path")
branches_grp_path = dbutils.widgets.get("branches_grp_path")
agencies_servicelines_branch_path = dbutils.widgets.get("agencies_servicelines_branch_path")
agencies_path = dbutils.widgets.get("agencies_path")
companies_path = dbutils.widgets.get("companies_path")
yrinvo_path = dbutils.widgets.get("yrinvo_path")
billing_notes_path = dbutils.widgets.get("billing_notes_path")
billing_notes_types_path = dbutils.widgets.get("billing_notes_types_path")
billing_note_comments_path = dbutils.widgets.get("billing_note_comments_path")
billing_note_reviews_path = dbutils.widgets.get("billing_note_reviews_path")
vendor_edi_filedata_path = dbutils.widgets.get("vendor_edi_filedata_path")
edi_fileimports_path = dbutils.widgets.get("edi_fileimports_path")
edi_filetypes_path = dbutils.widgets.get("edi_filetypes_path")
line_items_path = dbutils.widgets.get("line_items_path")
ppsresubmitebhistory_path = dbutils.widgets.get("ppsresubmitebhistory_path")
billing_submission_invoices_path = dbutils.widgets.get("billing_submission_invoices_path")
billing_submissions_path = dbutils.widgets.get("billing_submissions_path")
payments_path = dbutils.widgets.get("payments_path")
invoices_path = dbutils.widgets.get("invoices_path")
payor_src_path = dbutils.widgets.get("payor_src_path")
payor_types_path = dbutils.widgets.get("payor_types_path")
client_epi_fs_path = dbutils.widgets.get("client_epi_fs_path") 
client_epi_all_path = dbutils.widgets.get("client_epi_all_path")
client_epi_fs_insured_party_info_path = dbutils.widgets.get("client_epi_fs_insured_party_info_path") 
pdgm_period_path = dbutils.widgets.get("pdgm_period_path")
hchbofficemapping_path = dbutils.widgets.get("hchbofficemapping_path")
redarfact_path = dbutils.widgets.get("redarfact_path")

In [0]:
spark.sql(
    f"""
TRUNCATE TABLE {redarfact_path};
 """
)

In [0]:
spark.sql( 
        f""" 

CREATE OR REPLACE TEMPORARY VIEW temp_divisions AS
SELECT
  br.branch_code,
  br.branch_name,
  bg.bg_description AS Division
FROM {branches_table_path} br
INNER JOIN {branches_grp_branches_path} bgb 
  ON br.branch_code = bgb.bgb_branchcode
INNER JOIN {branches_grp_path} bg 
  ON bgb.bgb_bgid = bg.bg_id 
  AND bg.bg_description LIKE 'DIV%' 
  AND bg.bg_description NOT LIKE '%DIVISIONS%' 
  AND bg.bg_description NOT LIKE 'DIV: XX CLOSED OFFICES XX'
  AND bg.bg_active = 'Y';
  """ 
)

spark.sql( 
        f""" 
CREATE OR REPLACE TEMPORARY VIEW temp_areas AS
SELECT
  br.branch_code,
  br.branch_name,
  bg.bg_description AS Area
FROM {branches_table_path} br
INNER JOIN {branches_grp_branches_path} bgb 
  ON br.branch_code = bgb.bgb_branchcode
INNER JOIN {branches_grp_path} bg 
  ON bgb.bgb_bgid = bg.bg_id 
  AND bg.bg_description LIKE 'AREA%' 
  AND bg.bg_description NOT LIKE '%AREAS%'
  AND bg.bg_active = 'Y';
""" 
)

spark.sql(
    f""" 
CREATE OR REPLACE TEMPORARY VIEW temp_groups AS
SELECT
  br.branch_code,
  br.branch_name,
  bg.bg_description AS `Group`
FROM {branches_table_path} br
INNER JOIN {branches_grp_branches_path} bgb 
  ON br.branch_code = bgb.bgb_branchcode
INNER JOIN {branches_grp_path} bg 
  ON bgb.bgb_bgid = bg.bg_id 
  AND bg.bg_description LIKE 'GRP%' 
  AND bg.bg_description NOT LIKE '%GRPS%'
  AND bg.bg_active = 'Y';
    """ 
)

spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW temp_region AS
SELECT
  br.branch_code,
  br.branch_name,
  bg.bg_description AS Region
FROM {branches_table_path} br
INNER JOIN {branches_grp_branches_path} bgb 
  ON br.branch_code = bgb.bgb_branchcode
INNER JOIN {branches_grp_path} bg 
  ON bgb.bgb_bgid = bg.bg_id 
  AND bg.bg_description LIKE 'REG%' 
  AND bg.bg_description NOT LIKE '%REGS%'
  AND bg.bg_active = 'Y';
  """
)

spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW temp_branch_info AS
SELECT
  br.branch_code,
  br.branch_name,
  bg.bg_description AS Branch,
  c.company_name AS Company,
  br.branch_state
FROM {branches_table_path} br
INNER JOIN {branches_grp_branches_path} bgb 
  ON br.branch_code = bgb.bgb_branchcode
INNER JOIN {branches_grp_path} bg 
  ON bgb.bgb_bgid = bg.bg_id 
  AND bg.bg_description LIKE 'OFC%'
  AND bg.bg_active = 'Y'  
  AND bg.bg_description NOT LIKE 'OFC: *ALL HH OFFICES'
INNER JOIN {agencies_servicelines_branch_path} asb 
  ON asb.asb_branchcode = br.branch_code
INNER JOIN {agencies_path} a 
  ON a.Agency_id = asb.asb_agencyid
INNER JOIN {companies_path} c 
  ON c.company_id = a.agency_companyid 
  AND c.company_active = 'Y';
"""
)

spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW temp_invoices AS
SELECT InvNum
FROM {yrinvo_path}
WHERE StillOwe <> 0;
"""
)

spark.sql(
  f"""
  -- STEP 8: COMPUTE CLIENT BALANCES
CREATE OR REPLACE TEMPORARY VIEW temp_client_balances AS
SELECT 
  CltID, 
  SUM(invb.StillOwe) AS Balance
FROM {yrinvo_path} invb
INNER JOIN temp_invoices inv 
  ON invb.InvNum = inv.InvNum
GROUP BY CltID;
  """
)

spark.sql(
  f"""
  
-- STEP 9: GET LATEST BILLING NOTES
CREATE OR REPLACE TEMPORARY VIEW temp_billing_notes AS
SELECT
  bn.bn_invnum,
  MAX(COALESCE(bn.bn_id, 0)) AS bn_id
FROM {billing_notes_path} bn
INNER JOIN temp_invoices inv 
  ON bn.bn_invnum = inv.InvNum
GROUP BY bn.bn_invnum;
"""
)

spark.sql(
  f"""
  -- STEP 10: CREATE BILLING NOTE COUNT
CREATE OR REPLACE TEMPORARY VIEW temp_billing_note_count AS
SELECT 
  inv.InvNum,
  bn.bn_cltid,
  COUNT(DISTINCT bn.bn_id) AS BillingNoteCount 
FROM {billing_notes_path} bn
INNER JOIN temp_invoices inv 
  ON bn.bn_invnum = inv.InvNum
GROUP BY bn.bn_cltid, inv.InvNum;
"""
)

spark.sql(
  f"""
  -- STEP 11: PULL INVOICES WITH LAST BILLING NOTE INFO
CREATE OR REPLACE TEMPORARY VIEW temp_invoice_notes AS
SELECT
  inv.invnum,
  COALESCE(bn.bn_notes, 'No Notes') AS LastBillingNote,
  COALESCE(DATE_FORMAT(bn.bn_date, 'MM/dd/yyyy'), 'No Note') AS LastBillingNoteDate,
  COALESCE(DATE_FORMAT(bn.bn_date, 'HH:mm:ss.SSS'), 'No Note') AS LastBillingNoteTime,
  COALESCE(bn.bn_insertedby, 'No Note') AS LastBillingNoteUser,
  bnt1.bnt_desc AS LastBillingNoteType,
  bnc.bnc_desc AS LastBillingNoteCommentType,
  MAX(bnr.bnr_followupdate) AS LastFollowupDate
FROM temp_invoices inv
LEFT OUTER JOIN temp_billing_notes bnt 
  ON inv.InvNum = bnt.bn_invnum
LEFT OUTER JOIN {billing_notes_path} bn 
  ON bnt.bn_id = bn.bn_id
LEFT OUTER JOIN {billing_notes_types_path} bnt1 
  ON bn.bn_bntid = bnt1.bnt_id
LEFT OUTER JOIN {billing_note_comments_path} bnc 
  ON bn.bn_bncid = bnc.bnc_id
LEFT OUTER JOIN {billing_note_reviews_path} bnr 
  ON bnt.bn_id = bnr.bnr_bnid
GROUP BY
  inv.invnum,
  bn.bn_notes,
  bn.bn_date,
  bn.bn_insertedby,
  bnt1.bnt_desc,
  bnc.bnc_desc;
  """
)

spark.sql(
  f"""
  -- STEP 12: EXTRACT 835 EDI DATA
CREATE OR REPLACE TEMPORARY VIEW temp_835 AS
SELECT 
  `1` AS HCHBClaimNumber,
  MAX(vefd_vefid) AS LastImportID,
  MAX(vefd_sequence) AS LastSequence
FROM {vendor_edi_filedata_path}
INNER JOIN {edi_fileimports_path} 
  ON vefd_vefid = fi_importid
INNER JOIN {edi_filetypes_path}
  ON fi_ftid = ft_id
INNER JOIN {yrinvo_path} 
  ON CAST(InvNum AS STRING) = `1`
  AND InvNum > 0
WHERE ft_is835 = true
  AND vefd_segment = 'CLP'
GROUP BY `1`;
"""
)

spark.sql(
  f"""
  -- STEP 13: GET PAYOR CLAIM NUMBERS
CREATE OR REPLACE TEMPORARY VIEW temp_payor_claim_number AS
SELECT 
  c.HCHBClaimNumber, 
  vefd.`7` AS PayorClaimNumber   
FROM temp_835 c
INNER JOIN {vendor_edi_filedata_path} vefd
  ON c.HCHBClaimNumber = vefd.`1` 
  AND c.LastImportID = vefd.vefd_vefid 
  AND vefd.vefd_segment = 'CLP'
  AND c.LastSequence = vefd.vefd_sequence;
  """
)

spark.sql(
  f"""
  -- STEP 14: GET FIRST AND LAST SHIFT DATES
CREATE OR REPLACE TEMPORARY VIEW temp_first_shifts AS
SELECT 
  li.li_iid AS InvNum,
  MIN(li.li_servicedate) AS FirstShiftDate,
  MAX(li.li_servicedate) AS LastShiftDate
FROM {line_items_path} li
INNER JOIN temp_invoices inv 
  ON li.li_iid = inv.InvNum
GROUP BY li.li_iid;
"""
)

spark.sql(
  f"""
  
-- STEP 15: COLLECT ALL INVOICE SUBMISSIONS
CREATE OR REPLACE TEMPORARY VIEW temp_invoice_submissions AS
SELECT
  p.invnum,
  CASE
    WHEN ClaimType = 'S' THEN p.SubmitDate
    WHEN ClaimType = 'A' THEN p.SubmitDate
    WHEN ClaimType = 'C' THEN p.SubmitDate
    WHEN ClaimType = 'R' THEN p.ResubmissionDate
    ELSE p.SubmitDate
  END AS SubmissionDate
FROM {ppsresubmitebhistory_path} p
INNER JOIN temp_invoices il 
  ON p.InvNum = il.InvNum

UNION

SELECT
  bsi.bsi_invnum AS invnum,
  bs.bs_createdate AS SubmissionDate
FROM {billing_submission_invoices_path} bsi
INNER JOIN temp_invoices il 
  ON bsi.bsi_InvNum = il.InvNum       
INNER JOIN {billing_submissions_path} bs 
  ON bsi.bsi_bsid = bs.bs_id;
  """
)

spark.sql(
  f"""
  -- STEP 16: SUMMARIZE INVOICE SUBMISSIONS
CREATE OR REPLACE TEMPORARY VIEW temp_invoice_submission_summary AS
SELECT
  Invnum,
  MIN(DATE(SubmissionDate)) AS FirstSubmissionDate,
  MAX(DATE(SubmissionDate)) AS LastSubmissionDate
FROM temp_invoice_submissions
GROUP BY Invnum;
"""
)

spark.sql(
  f"""
  -- STEP 17: GET LAST PAYMENT APPLIED
CREATE OR REPLACE TEMPORARY VIEW temp_invoice_last_payment AS
SELECT 
  hp.p_invoiceid, 
  MAX(hp.p_postdate) AS LastPaymentApplied
FROM {payments_path} hp
INNER JOIN temp_invoices inv 
  ON hp.p_invoiceid = inv.InvNum
GROUP BY hp.p_invoiceid;
"""
)

spark.sql(
  f"""
  -- STEP 18: GET INVOICE POST/BILL DATE
CREATE OR REPLACE TEMPORARY VIEW temp_invoice_post_date AS
SELECT
  bi.i_id, 
  bi.i_postdate AS BillDate
FROM {invoices_path} bi
INNER JOIN temp_invoices inv 
  ON inv.InvNum = bi.i_id;
  """
)

In [0]:
spark.sql(
    f"""
    -- STEP 21: INSERT DATA INTO TARGET TABLE
INSERT INTO {redarfact_path} 
SELECT DISTINCT 
  inv.InvNum AS invnum,
  
  -- Client Balance
  CAST(cb.balance AS DECIMAL(18,2)) AS clientbalance,
  
  -- Episode Info
  epi.epi_StartOfEpisode AS episodestart,
  epi.epi_EndOfEpisode AS episodeend,
  
  -- Billing Notes
  bnc.BillingNoteCount AS billingnotecount,
  iss.FirstSubmissionDate AS firstsubmissiondate,
  DATE_FORMAT(invlp.LastPaymentApplied, 'MM/dd/yyyy') AS lastpaymentapplied,
  DATE_FORMAT(fs.FirstShiftDate, 'MM/dd/yyyy') AS firstdos,
  DATE_FORMAT(fs.LastShiftDate, 'MM/dd/yyyy') AS lastdos,
  
  -- Financial Data
  inv.TotChg AS charges,
  inv.TotChg - inv.StillOwe AS payments,
  inv.StillOwe AS stillowe,
  
  -- Define Quarter End Date (adjust dates as needed for your fiscal calendar)
  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) <= 60 
    THEN inv.StillOwe
    ELSE 0
  END AS days_0_60,

  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 61 AND 90 
    THEN inv.StillOwe
    ELSE 0
  END AS days_61_90,

  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 91 AND 120 
    THEN inv.StillOwe
    ELSE 0
  END AS days_91_120,

  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 121 AND 150 
    THEN inv.StillOwe
    ELSE 0
  END AS days_121_150,

  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 151 AND 180 
    THEN inv.StillOwe
    ELSE 0
  END AS days_151_180,

  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) >= 181
    THEN inv.StillOwe
    ELSE 0
  END AS days_181_plus,

  -- Billing Note Details
  invn.LastBillingNote AS lastbillingnote,
  invn.LastBillingNoteDate AS lastbillingnotedate,
  COALESCE(invn.LastBillingNoteType, 'No Note') AS lastbillingnotetype,
  COALESCE(invn.LastBillingNoteCommentType, 'No Note') AS lastbillingnotecommenttype,
  invn.LastFollowupDate AS lastfollowupdate,
  invn.LastBillingNoteUser AS lastbillingnoteuser,
  
  -- Payor Info
  pt.pt_desc AS payortype, 
  ps.ps_id,
  pcn.PayorClaimNumber AS payorclaimnumber,
  
  -- Branch Info
  bi.Branch AS branchname,
  inv.BranchID AS branchid,
  
  -- Episode ID
  epi.epi_id AS episodeid,
  
  -- Billing Frequency
  CASE inv.freq
    WHEN 0 THEN 'On Demand'
    WHEN 5 THEN 'Episodic'
    WHEN 6 THEN 'Per Diem'
    ELSE 'Unknown'
  END AS billingfrequency,
  
  -- Extended Aging Buckets
  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 181 AND 270
    THEN inv.StillOwe
    ELSE 0
  END AS days_181_270,
  
  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 271 AND 360
    THEN inv.StillOwe
    ELSE 0
  END AS days_271_360,
  
  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 361 AND 450
    THEN inv.StillOwe
    ELSE 0
  END AS days_361_450,
  
  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) BETWEEN 451 AND 540
    THEN inv.StillOwe
    ELSE 0
  END AS days_451_540,
  
  CASE 
    WHEN DATEDIFF(
      CASE 
        WHEN CURRENT_DATE BETWEEN '2024-01-05' AND '2024-04-04' THEN '2024-03-31'
        WHEN CURRENT_DATE BETWEEN '2024-04-05' AND '2024-07-04' THEN '2024-06-30'
        WHEN CURRENT_DATE BETWEEN '2024-07-05' AND '2024-10-03' THEN '2024-09-29'
        WHEN CURRENT_DATE BETWEEN '2024-10-04' AND '2025-01-02' THEN '2024-12-29'
        WHEN CURRENT_DATE BETWEEN '2025-01-03' AND '2025-04-03' THEN '2025-03-30'
        WHEN CURRENT_DATE BETWEEN '2025-04-04' AND '2025-07-03' THEN '2025-06-29'
        WHEN CURRENT_DATE BETWEEN '2025-07-04' AND '2025-10-02' THEN '2025-09-28'
        WHEN CURRENT_DATE BETWEEN '2025-10-03' AND '2026-01-01' THEN '2025-12-28'
        ELSE '2025-12-28'
      END,
      CASE inv.Freq 
        WHEN 5 THEN epi.epi_EndOfEpisode
        ELSE fs.firstShiftDate
      END
    ) >= 541
    THEN inv.StillOwe
    ELSE 0
  END AS days_541_plus,
  
  -- Bill Date and PDGM Period
  ipd.BillDate AS billdate,
  inv.PdgmPeriodId AS pdgmperiodid,
  
  -- Period Start and End (NULL placeholders - add logic if these columns exist in source)
  NULL AS periodstart,
  NULL AS periodend

FROM {yrinvo_path} inv
INNER JOIN temp_invoices il 
  ON inv.InvNum = il.InvNum
LEFT OUTER JOIN {payor_src_path} ps 
  ON inv.GroupID = ps.ps_id
LEFT OUTER JOIN {payor_types_path} pt 
  ON ps.ps_ptid = pt.pt_id
LEFT OUTER JOIN {client_epi_fs_path} cefs 
  ON inv.PrimID = cefs.cefs_id
LEFT OUTER JOIN {client_epi_all_path} epi 
  ON cefs.cefs_epiid = epi.epi_id
LEFT OUTER JOIN temp_areas area 
  ON inv.BranchID = area.branch_code
LEFT OUTER JOIN temp_branch_info bi 
  ON inv.branchid = bi.branch_code
LEFT OUTER JOIN temp_divisions div 
  ON inv.BranchID = div.branch_code
LEFT OUTER JOIN temp_region reg 
  ON inv.BranchID = reg.branch_code 
LEFT OUTER JOIN temp_client_balances cb 
  ON inv.CltID = cb.CltID
LEFT OUTER JOIN temp_billing_note_count bnc 
  ON inv.InvNum = bnc.InvNum
LEFT OUTER JOIN temp_invoice_notes invn 
  ON inv.InvNum = invn.invnum
LEFT OUTER JOIN temp_payor_claim_number pcn 
  ON inv.InvNum = pcn.HCHBClaimNumber
LEFT OUTER JOIN {client_epi_fs_insured_party_info_path} AS cefsipi 
  ON cefsipi.cefsipi_cefsid = cefs.cefs_id 
LEFT OUTER JOIN temp_first_shifts fs 
  ON inv.InvNum = fs.InvNum
LEFT OUTER JOIN temp_invoice_submission_summary iss 
  ON inv.InvNum = iss.Invnum
LEFT OUTER JOIN temp_invoice_last_payment invlp 
  ON inv.InvNum = invlp.p_invoiceid
LEFT OUTER JOIN temp_invoice_post_date ipd 
  ON inv.InvNum = ipd.i_id
WHERE SUBSTRING(bi.branch, 13, 1) <> 'Q' OR bi.branch IS NULL;

  """
)

In [0]:
spark.sql(
    f""" 
-- Update 1: Set PeriodStart and PeriodEnd
MERGE INTO {redarfact_path}  AS rar
USING {pdgm_period_path} AS pp
ON pp.pp_id = rar.pdgmPeriodId
WHEN MATCHED THEN
  UPDATE SET
    rar.periodstart = pp.pp_startDate,
    rar.periodend = pp.pp_endDate;
    """
)
spark.sql(
    f""" 
-- Update 2: Update BranchID from mapping table
MERGE INTO {redarfact_path}  AS ha
USING {hchbofficemapping_path} AS om
ON ha.branchid = om.SourceOfficeCode
  AND ha.branchid RLIKE '[A-Z]'
WHEN MATCHED THEN
  UPDATE SET ha.branchid = om.TargetOfficeNumber;
  """
)
spark.sql(
    f""" 
-- Update 3: Update specific BranchID 644 to 9644
UPDATE {redarfact_path} 
SET branchid = '9644'
WHERE branchid = '644';
"""
)
spark.sql(
    f""" 
-- Update 4: Update specific BranchID 701 to 9701
UPDATE {redarfact_path} 
SET branchid = '9701'
WHERE branchid = '701';
"""
)